In [1]:
# 01_build_panel.ipynb
# Build a person-year analytic panel from Korea Health Panel (2nd phase, 2019-2024).
# Constructs out-of-pocket (OOP) spending, catastrophic health expenditure (CHE),
# acute/chronic health-shock indicators, private health insurance (PHI) holding,
# and treatment-cohort variables. Output is written to ../data/processed/.

import warnings, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, pyreadstat

RAW = "../data/raw"           # relative paths only (no absolute paths are printed)
PROC = "../data/processed"
TAB = "../results/tables"
os.makedirs(PROC, exist_ok=True); os.makedirs(TAB, exist_ok=True)
WAVES = [(2019, "a"), (2020, "b"), (2021, "c"), (2022, "d"), (2023, "e"), (2024, "f")]

def num(s):
    return pd.to_numeric(s, errors="coerce")

# ---- read each wave and aggregate medical-service events to person-year ----
frames = []
for yr, pfx in WAVES:
    ms, _ = pyreadstat.read_sas7bdat(f"{RAW}/sas_{yr}/{pfx}_ms.sas7bdat",
                                     usecols=["HHID", "PIDWON", "MS1", "MEXP3_1", "MEXP4_1", "DAYS"])
    fac = num(ms["MEXP3_1"]).where(lambda x: x >= 0)   # -9 is a non-response code -> missing
    pha = num(ms["MEXP4_1"]).where(lambda x: x >= 0)
    ms["oop"] = fac.fillna(0) + pha.fillna(0)          # OOP = facility payment + pharmacy payment
    ms["ms1"] = num(ms["MS1"]); ms["days"] = num(ms["DAYS"]).where(lambda x: x >= 0)
    g = ms.groupby("PIDWON").agg(
        oop=("oop", "sum"), n_visit=("oop", "size"),
        n_er=("ms1", lambda s: (s == 1).sum()),
        n_inp=("ms1", lambda s: (s == 2).sum()),
        n_out=("ms1", lambda s: (s == 3).sum()),
        inp_days=("days", lambda s: s[ms.loc[s.index, "ms1"] == 2].sum())).reset_index()

    ind, _ = pyreadstat.read_sas7bdat(f"{RAW}/sas_{yr}/{pfx}_ind.sas7bdat",
        usecols=["HHID", "PIDWON", "SEX", "BIRTH_Y", "EDU", "MARR", "ECO1", "HEALTH_INS",
                 "DISA_YN", "CD", "CD1_HTN", "CD1_DM", "H_INC_TOT", "REGION1", "I_WGC", "I_WSC"])
    for c in ["SEX", "BIRTH_Y", "EDU", "MARR", "ECO1", "HEALTH_INS", "DISA_YN", "CD",
              "CD1_HTN", "CD1_DM", "H_INC_TOT", "REGION1", "I_WGC", "I_WSC"]:
        ind[c] = num(ind[c])
    ind["age"] = yr - ind["BIRTH_Y"]
    ind["hh_inc_won"] = ind["H_INC_TOT"] * 10000.0     # household income is in units of 10,000 KRW
    ind["chronic"] = (ind["CD"] == 1).astype("Int64")   # 1 = has chronic disease
    ind["htn"] = (ind["CD1_HTN"] == 1).astype("Int64")
    ind["dm"] = (ind["CD1_DM"] == 1).astype("Int64")
    ind["female"] = (ind["SEX"] == 2).astype("Int64")
    ind["disabled"] = (ind["DISA_YN"] == 1).astype("Int64")

    phi, _ = pyreadstat.read_sas7bdat(f"{RAW}/sas_{yr}/{pfx}_phi.sas7bdat",
                                      usecols=["HHID", "PHI_PID", "PHI6"])
    phi["PHI_PID"] = num(phi["PHI_PID"]); phi["prem"] = num(phi["PHI6"]).where(lambda x: x >= 0)
    pers_phi = phi.groupby("PHI_PID").agg(phi_n=("PHI_PID", "size"),
              phi_prem=("prem", "sum")).reset_index().rename(columns={"PHI_PID": "PIDWON"})
    hh_phi = phi.groupby("HHID").agg(hh_phi_n=("PHI_PID", "size"),
              hh_phi_prem=("prem", "sum")).reset_index()

    d = ind.merge(g, on="PIDWON", how="left").merge(pers_phi, on="PIDWON", how="left").merge(hh_phi, on="HHID", how="left")
    for c in ["oop", "n_visit", "n_er", "n_inp", "n_out", "inp_days", "phi_n", "phi_prem", "hh_phi_n", "hh_phi_prem"]:
        d[c] = d[c].fillna(0)
    d["any_visit"] = (d["n_visit"] > 0).astype(int)
    d["phi_any"] = (d["phi_n"] > 0).astype(int)
    d["hh_phi_any"] = (d["hh_phi_n"] > 0).astype(int)
    d["year"] = yr
    frames.append(d)

panel = pd.concat(frames, ignore_index=True)

# ---- household-year OOP and catastrophic health expenditure (CHE) ----
hh_oop = panel.groupby(["HHID", "year"]).agg(hh_oop=("oop", "sum"), hh_size=("PIDWON", "size")).reset_index()
panel = panel.merge(hh_oop, on=["HHID", "year"], how="left")
hh_inc = panel.groupby(["HHID", "year"])["hh_inc_won"].max().reset_index().rename(columns={"hh_inc_won": "hh_inc_won_hh"})
panel = panel.merge(hh_inc, on=["HHID", "year"], how="left")
cap = panel["hh_inc_won_hh"].replace(0, np.nan)        # capacity to pay (household income)
panel["che_ratio"] = panel["hh_oop"] / cap
for th in [0.10, 0.25, 0.40]:
    panel[f"che{int(th*100)}"] = (panel["che_ratio"] >= th).astype("Int64")
    panel.loc[cap.isna(), f"che{int(th*100)}"] = pd.NA

# ---- longitudinal health-shock indicators (require previous observed wave) ----
panel = panel.sort_values(["PIDWON", "year"]).reset_index(drop=True)
panel["chronic_lag"] = panel.groupby("PIDWON")["chronic"].shift(1)
panel["ninp_lag"] = panel.groupby("PIDWON")["n_inp"].shift(1)
panel["shock_newchronic"] = ((panel["chronic"] == 1) & (panel["chronic_lag"] == 0)).astype("Int64")
panel["shock_newhosp"] = ((panel["n_inp"] > 0) & (panel["ninp_lag"] == 0)).astype("Int64")
panel.loc[panel["chronic_lag"].isna(), "shock_newchronic"] = pd.NA
panel.loc[panel["ninp_lag"].isna(), "shock_newhosp"] = pd.NA
panel["shock_any"] = (((panel["shock_newchronic"] == 1) | (panel["shock_newhosp"] == 1))).astype("Int64")
panel.loc[panel["shock_newchronic"].isna() & panel["shock_newhosp"].isna(), "shock_any"] = pd.NA

# ---- treatment cohort (first acute shock = first new hospitalization) and baseline PHI ----
first = panel[panel["shock_newhosp"] == 1].groupby("PIDWON")["year"].min().rename("cohort")
panel = panel.merge(first, on="PIDWON", how="left")
panel["cohort"] = panel["cohort"].fillna(0).astype(int)   # 0 marks never-treated
base_yr = panel.groupby("PIDWON")["year"].transform("min")
base_phi = panel[panel["year"] == base_yr].groupby("PIDWON")["hh_phi_any"].first().rename("base_phi")
panel = panel.merge(base_phi, on="PIDWON", how="left")

# ---- derived analytic variables ----
panel["inc_q"] = panel.groupby("year")["hh_inc_won_hh"].transform(lambda s: pd.qcut(s.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]))
panel["ihs_oop"] = np.arcsinh(panel["oop"])
panel["log_hhinc"] = np.log(panel["hh_inc_won_hh"].clip(lower=1))

panel.to_parquet(f"{PROC}/khp_panel.parquet", index=False)

# ---- panel-structure summary table ----
struct = panel.groupby("year").agg(person_years=("PIDWON", "size"),
          with_visit=("any_visit", "sum"), phi_rate=("hh_phi_any", "mean")).reset_index()
nyr = panel.groupby("PIDWON")["year"].nunique()
struct_extra = pd.DataFrame({"metric": ["distinct_persons", "balanced_6y_persons", "total_person_years"],
          "value": [panel["PIDWON"].nunique(), int((nyr == 6).sum()), len(panel)]})
struct.to_csv(f"{TAB}/t01_panel_structure.csv", index=False)
struct_extra.to_csv(f"{TAB}/t01_panel_summary.csv", index=False)

print("Panel built. person-years = {:,} | distinct persons = {:,} | balanced 6y = {:,}".format(
    len(panel), panel["PIDWON"].nunique(), int((nyr == 6).sum())))
print(struct.to_string(index=False))


Panel built. person-years = 86,569 | distinct persons = 21,651 | balanced 6y = 11,225
 year  person_years  with_visit  phi_rate
 2019         16587       12661  0.814071
 2020         14844       11941  0.819995
 2021         13799       12183  0.826074
 2022         12909       11340  0.816097
 2023         12169       10867  0.827266
 2024         16261       14582  0.812250
